In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r'C:\Users\bharg\wind_power_forecasting\data\T1.csv')

In [4]:

df['Date/Time'] = pd.to_datetime(df['Date/Time'], format='%d %m %Y %H:%M')
df.set_index('Date/Time', inplace=True)
df.rename(columns={
    'LV ActivePower (kW)': 'ActivePower',
    'Wind Speed (m/s)': 'WindSpeed',
    'Theoretical_Power_Curve (KWh)': 'TheoreticalPower',
    'Wind Direction (°)': 'WindDirection'
}, inplace=True)

df['ActivePower'] = df['ActivePower'].clip(lower=0)

In [5]:
# 1. Time Features
df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)

In [6]:
# Cyclical encoding (Very Important for time-series)
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12)

In [7]:
# 2. Wind Physics Features (This shows domain understanding)
df['WindSpeed_squared'] = df['WindSpeed'] ** 2
df['WindSpeed_cubed']   = df['WindSpeed'] ** 3
df['WindSpeed_4']       = df['WindSpeed'] ** 4

# Wind Direction Components
df['WindDir_rad'] = np.deg2rad(df['WindDirection'])
df['WindDir_sin'] = np.sin(df['WindDir_rad'])
df['WindDir_cos'] = np.cos(df['WindDir_rad'])

In [8]:
# 3. Lag & Rolling Features
for lag in [1, 2, 3, 6, 12]:  # 10min to 2 hours
    df[f'WindSpeed_lag_{lag}'] = df['WindSpeed'].shift(lag)
    df[f'ActivePower_lag_{lag}'] = df['ActivePower'].shift(lag)

# Rolling statistics
for window in [6, 12, 36]:   # 1h, 2h, 6h
    df[f'WindSpeed_roll_mean_{window}'] = df['WindSpeed'].rolling(window).mean()
    df[f'WindSpeed_roll_std_{window}'] = df['WindSpeed'].rolling(window).std()

In [9]:
# 4. Turbine Performance Features
df['PowerCoefficient'] = df['ActivePower'] / (df['TheoreticalPower'] + 1e-6)
df['Efficiency'] = np.where(df['TheoreticalPower'] > 0, 
                           df['ActivePower'] / df['TheoreticalPower'], 0)

# 5. Wind Speed Bins (Operational Regions)
df['WindSpeed_bin'] = pd.cut(df['WindSpeed'], 
                            bins=[0, 3, 12, 25, 50], 
                            labels=['Low', 'Optimal', 'High', 'Extreme'])

print("✅ Feature Engineering Completed!")

✅ Feature Engineering Completed!


In [10]:
print("Total Features:", df.shape[1])

Total Features: 37
